# Practical 4: Label Requests as Benign or Attack (Basic Classification)

**Objective:** Create labeled categories for analysis.

- Use known attack patterns (SQL injection, path traversal, brute-force login) to label logs
- Create a new column: `label` -> `benign`, `sqli`, `path_traversal`, `brute_force`

In [1]:
import pandas as pd
import re

df = pd.read_csv('logs/cleaned_access_log.csv', parse_dates=['Date/Time'])
print('Rows loaded:', len(df))
df.head()

Rows loaded: 47685


,IP Address,Date/Time,Request Type,Resource,Protocol,Status Code,Bytes Sent,Referrer,User Agent,Resource Normalized
0,144.187.77.221,2026-08-01 00:00:25+05:30,GET,/api/products,HTTP/1.1,404,418,https://facebook.com,mozilla/5.0 (iphone; cpu iphone os 17_0 like m...,/api/products
1,94.87.216.251,2026-08-01 00:02:59+05:30,GET,/index.html,HTTP/1.1,304,8710,https://bing.com,mozilla/5.0 (windows nt 10.0; win64; x64) appl...,/index.html
2,192.168.1.168,2026-08-01 00:03:15+05:30,GET,/about,HTTP/1.1,200,463,direct,mozilla/5.0 (windows nt 10.0; win64; x64) appl...,/about
3,72.23.185.21,2026-08-01 00:03:16+05:30,GET,/images/logo.png,HTTP/1.1,200,2243,https://twitter.com,mozilla/5.0 (windows nt 10.0; win64; x64) appl...,/images/logo.png
4,176.253.49.242,2026-08-01 00:04:04+05:30,POST,/style.css,HTTP/1.1,200,1414,https://example.com,mozilla/5.0 (windows nt 10.0; win64; x64) appl...,/style.css


## 1. Define attack signature patterns

**SQL injection** — quote characters, boolean tautologies (`OR 1=1`), SQL keywords (`UNION`, `SELECT`, `DROP`), or timing attacks (`SLEEP(`).

**Path traversal** — literal `../` or its URL-encoded forms (`%2e%2e%2f`, `..%2f`).

In [2]:
sqli_pattern = re.compile(
    r"(\%27|'|--|\bunion\b|\bselect\b|\bdrop\b|\bsleep\s*\()",
    re.IGNORECASE
)

traversal_pattern = re.compile(
    r"(\.\.\/|\.\.%2f|%2e%2e)",
    re.IGNORECASE
)

def pattern_label(resource: str):
    if sqli_pattern.search(resource):
        return 'sqli'
    if traversal_pattern.search(resource):
        return 'path_traversal'
    return None

df['pattern_label'] = df['Resource'].astype(str).apply(pattern_label)
df['pattern_label'].value_counts(dropna=False)

pattern_label
None              45718
path_traversal     1000
sqli                967
Name: count, dtype: int64

## 2. Detect brute-force login attempts

A brute-force pattern is *behavioral*, not visible in a single row — it's many `POST /login` requests from the **same IP** in a short window. We flag any IP that made more login attempts than a normal user reasonably would.

In [3]:
BRUTE_FORCE_THRESHOLD = 5  # more than this many login POSTs from one IP looks automated

login_attempts = df[(df['Request Type'] == 'POST') & (df['Resource'] == '/login')]
attempts_per_ip = login_attempts.groupby('IP Address').size()

brute_force_ips = attempts_per_ip[attempts_per_ip > BRUTE_FORCE_THRESHOLD].index
print(f'IPs flagged as brute-force sources: {len(brute_force_ips)}')
attempts_per_ip[attempts_per_ip > BRUTE_FORCE_THRESHOLD].sort_values(ascending=False).head(10)

IPs flagged as brute-force sources: 150


IP Address
56.7.5.9          25
93.89.37.44       25
221.82.246.172    25
41.203.107.144    25
201.126.226.82    25
139.11.49.144     25
160.49.247.216    24
163.204.178.72    24
164.110.122.80    24
166.105.39.33     24
dtype: int64

## 3. Combine into a single `label` column

Priority order: an explicit SQLi/traversal signature in the URL is unambiguous, so it wins even on a login-page request. Otherwise, if the row is a login POST from a flagged IP, it's brute_force. Everything else is benign.

In [4]:
def final_label(row):
    # NOTE: an empty pattern_label comes back as NaN (a float), not Python's None,
    # so we must check with pd.notna() here rather than 'is not None' — otherwise
    # every benign row would incorrectly get labeled NaN instead of 'benign'.
    if pd.notna(row['pattern_label']):
        return row['pattern_label']
    if row['Request Type'] == 'POST' and row['Resource'] == '/login' and row['IP Address'] in brute_force_ips:
        return 'brute_force'
    return 'benign'

df['label'] = df.apply(final_label, axis=1)
df = df.drop(columns=['pattern_label'])
df['label'].value_counts()

label
benign            43233
brute_force        2485
path_traversal     1000
sqli                967
Name: count, dtype: int64

## 4. Sanity-check a few labeled examples

In [5]:
for lbl in df['label'].unique():
    print(f"\n--- {lbl} ---")
    print(df[df['label'] == lbl][['IP Address', 'Request Type', 'Resource', 'Status Code']].head(3).to_string(index=False))


--- benign ---
    IP Address Request Type      Resource  Status Code
144.187.77.221          GET /api/products          404
 94.87.216.251          GET   /index.html          304
 192.168.1.168          GET        /about          200

--- sqli ---
    IP Address Request Type                                                              Resource  Status Code
  90.1.220.176          GET /search?q=%27%20UNION%20SELECT%20username%2Cpassword%20FROM%20users--          403
118.254.124.99          GET                                                    /login?user=admin'          500
 113.72.67.208          GET /search?q=%27%20UNION%20SELECT%20username%2Cpassword%20FROM%20users--          403

--- path_traversal ---
    IP Address Request Type                                   Resource  Status Code
  124.54.1.198          GET               /..%2f..%2f..%2fetc%2fpasswd          403
  35.58.63.233          GET             /images/../../../../etc/shadow          403
209.200.28.134          GET /d

## Observations

- Label distribution is heavily imbalanced toward `benign`, which mirrors real traffic. This is exactly the imbalance Practical 6 (SMOTE/oversampling) will need to address.
- `brute_force` is derived from **behavior across multiple rows** (repeated login attempts per IP) rather than a single request's content, unlike the other three labels.
- The threshold of 5 attempts is a simplifying assumption for this basic classifier; a production system would also weigh the *time window* between attempts, not just the raw count.

## Save labeled dataset for Practical 5

In [6]:
df.to_csv('logs/labeled_access_log.csv', index=False)
print('Saved logs/labeled_access_log.csv')

Saved logs/labeled_access_log.csv
